# Hermon SQL Query Experiments

Use this notebook to experiment with SQL safely against the local Postgres database.

This notebook uses `from app.db import get_db`, so every query goes through the same read-only validation layer that the agent will use.

**Money unit warning:** source revenue tables store amounts in minor units, so business-facing source-table queries must divide fields such as `payments.amount`, `refunds.amount`, and `contracts.total_value` by `100.0`. `diagnostic_lead_snapshot` money fields are already major-unit EUR values and must not be divided again. Some historical output cells in this notebook may show raw minor-unit amounts from older experiments.

## 1. Setup

Run this first. It loads `.env`, imports `get_db()`, and sets display options.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for path in [start, *start.parents]:
        if (path / ".env").exists():
            return path
    raise FileNotFoundError("Could not find .env in the current folder or parent folders.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from app.db import get_db

ORG_ID = os.getenv("HERMON_DEFAULT_CLERK_ORG_ID")
MAX_ROWS = int(os.getenv("HERMON_SQL_MAX_ROWS", "500"))

assert ORG_ID, "Missing HERMON_DEFAULT_CLERK_ORG_ID in .env"

db = get_db()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 180)

print(f"Project root: {PROJECT_ROOT}")
print(f"Default org:  {ORG_ID}")
print(f"Max rows:     {MAX_ROWS}")

Project root: /Users/mitulkanani/Desktop/Projects/Hermon_Agentic_Chatbot
Default org:  org_3ARuGHeqbbEu5FNexlpC7ElaiyW
Max rows:     500


## 2. Query Helper

`q()` runs SQL through `app.db.ReadOnlyPostgres` and returns a DataFrame.

The helper enforces read-only SQL, blocks dangerous tables/columns, requires `clerk_org_id` for business tables, and caps row output.

In [2]:
def q(sql: str, params: dict | None = None, limit: int | None = MAX_ROWS) -> pd.DataFrame:
    """Run SQL through the safe app.db helper and return a DataFrame."""
    return db.query_df(sql, params=params or {}, max_rows=limit)


def org_params(**extra) -> dict:
    """Default params for org-scoped business queries."""
    return {"org_id": ORG_ID, **extra}

## 3. Connection Test

In [3]:
connection_test_df = q("""
SELECT
  current_database() AS database_name,
  current_user AS database_user,
  NOW() AS checked_at
""", limit=5)

connection_test_df

,database_name,database_user,checked_at
0,hermon_explore,mitulkanani,2026-04-29 13:02:34.881687+00:00


## 4. Database Tables

In [4]:
tables_df = q("""
SELECT
  table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_type = 'BASE TABLE'
ORDER BY table_name
""", limit=None)

tables_df

,table_name
0,_prisma_migrations
1,appointment_event_types
2,appointments
3,audit_logs
4,calendar_availability_snapshots
5,contract_subscriptions
6,contracts
7,fathom_call_records
8,fathom_org_settings
9,integration_health_checks


## 5. Row Counts For Main Business Tables

In [5]:
business_counts_df = q("""
SELECT 'leads' AS table_name, COUNT(*) AS active_rows FROM leads WHERE clerk_org_id = :org_id AND is_deleted = false
UNION ALL
SELECT 'sales_statuses', COUNT(*) FROM sales_statuses WHERE clerk_org_id = :org_id
UNION ALL
SELECT 'opt_ins', COUNT(*) FROM opt_ins WHERE clerk_org_id = :org_id
UNION ALL
SELECT 'appointments', COUNT(*) FROM appointments WHERE clerk_org_id = :org_id AND is_deleted = false
UNION ALL
SELECT 'lead_notes', COUNT(*) FROM lead_notes WHERE clerk_org_id = :org_id AND is_deleted = false
UNION ALL
SELECT 'fathom_call_records', COUNT(*) FROM fathom_call_records WHERE clerk_org_id = :org_id
UNION ALL
SELECT 'programs', COUNT(*) FROM programs WHERE clerk_org_id = :org_id AND is_deleted = false
UNION ALL
SELECT 'contracts', COUNT(*) FROM contracts WHERE clerk_org_id = :org_id AND is_deleted = false
UNION ALL
SELECT 'payments', COUNT(*) FROM payments WHERE clerk_org_id = :org_id AND is_deleted = false
UNION ALL
SELECT 'refunds', COUNT(*) FROM refunds WHERE clerk_org_id = :org_id
ORDER BY active_rows DESC
""", org_params(), limit=None)

business_counts_df

,table_name,active_rows
0,lead_notes,1510
1,opt_ins,750
2,leads,515
3,appointments,451
4,payments,212
5,contracts,181
6,fathom_call_records,93
7,sales_statuses,10
8,programs,5
9,refunds,1


## 6. Leads By Status

In [6]:
leads_by_status_df = q("""
SELECT
  COALESCE(ss.name, 'No Status') AS status_name,
  ss.role AS status_role,
  COUNT(*) AS lead_count
FROM leads l
LEFT JOIN sales_statuses ss
  ON ss.id = l.status_id
 AND ss.clerk_org_id = l.clerk_org_id
WHERE l.clerk_org_id = :org_id
  AND l.is_deleted = false
GROUP BY COALESCE(ss.name, 'No Status'), ss.role
ORDER BY lead_count DESC
""", org_params(), limit=None)

leads_by_status_df

,status_name,status_role,lead_count
0,Won,WON,134
1,New Lead,NEW_LEAD,96
2,No Sale - Lost,LOST,62
3,Canceled,CANCELED,59
4,No Show,NO_SHOW,47
5,No Sale - Follow Up,FOLLOW_UP,46
6,Call Booked,APPOINTMENT_BOOKED,37
7,Rescheduled,RESCHEDULED,22
8,No Sale - Unqualified,UNQUALIFIED,10
9,Deposit,PARTIAL_PAYMENT,2


## 7. Leads By Source

In [7]:
leads_by_source_df = q("""
SELECT
  source,
  first_source_name,
  last_source_name,
  COUNT(*) AS lead_count
FROM leads
WHERE clerk_org_id = :org_id
  AND is_deleted = false
GROUP BY source, first_source_name, last_source_name
ORDER BY lead_count DESC
""", org_params(), limit=100)

leads_by_source_df

,source,first_source_name,last_source_name,lead_count
0,CALENDLY,NaN,NaN,232
1,OTHER,NaN,NaN,120
2,OTHER,youtube,youtube,79
3,OTHER,instagram,instagram,22
4,OTHER,YouTube,YouTube,19
5,OTHER,aff,aff,16
6,MANUAL,NaN,NaN,12
7,CALENDLY,youtube,youtube,5
8,OTHER,Instagram,Instagram,4
9,OTHER,affiliate,affiliate,2


## 8. Payment Summary

In [8]:
payments_by_status_df = q("""
SELECT
  status,
  payment_provider,
  currency,
  COUNT(*) AS payment_count,
  SUM(amount) AS total_amount,
  COUNT(*) FILTER (WHERE paid_at IS NOT NULL) AS payments_with_paid_at,
  COUNT(*) FILTER (WHERE due_date IS NOT NULL) AS payments_with_due_date
FROM payments
WHERE clerk_org_id = :org_id
  AND is_deleted = false
GROUP BY status, payment_provider, currency
ORDER BY status, payment_provider
""", org_params(), limit=None)

payments_by_status_df

,status,payment_provider,currency,payment_count,total_amount,payments_with_paid_at,payments_with_due_date
0,DRAFT,WHOP,eur,4,400000.0,0,4
1,PENDING,WHOP,eur,28,10195000.0,0,28
2,PENDING,MANUAL,eur,2,730000.0,0,2
3,PAID,WHOP,eur,109,37815000.0,109,109
4,PAID,MANUAL,eur,49,18610000.0,49,49
5,FAILED,WHOP,eur,4,905000.0,0,4
6,LOST,WHOP,eur,12,4070000.0,0,12
7,LOST,MANUAL,eur,3,830100.0,0,3
8,REFUNDED,MANUAL,eur,1,350000.0,1,1


## 9. Contract Summary

In [9]:
contracts_by_status_df = q("""
SELECT
  c.status,
  c.type,
  c.currency,
  p.name AS program_name,
  COUNT(*) AS contract_count,
  SUM(c.total_value) AS total_contract_value,
  COUNT(*) FILTER (WHERE c.sent_at IS NOT NULL) AS sent_count,
  COUNT(*) FILTER (WHERE c.signed_at IS NOT NULL) AS signed_count
FROM contracts c
JOIN programs p
  ON p.id = c.program_id
 AND p.clerk_org_id = c.clerk_org_id
WHERE c.clerk_org_id = :org_id
  AND c.is_deleted = false
  AND p.is_deleted = false
GROUP BY c.status, c.type, c.currency, p.name
ORDER BY contract_count DESC
""", org_params(), limit=None)

contracts_by_status_df

,status,type,currency,program_name,contract_count,total_contract_value,sent_count,signed_count
0,SIGNED,PIF,eur,Freedom Academy,135,57690000.0,135,135
1,DRAFT,PIF,eur,Freedom Academy,28,12010000.0,0,0
2,VOIDED,PIF,eur,Freedom Academy,13,5120100.0,10,3
3,SENT,PIF,eur,Freedom Academy,2,980000.0,2,0
4,VOIDED,PP,eur,Freedom Academy,1,500000.0,1,0
5,SIGNED,PP,eur,Freedom Academy,1,350000.0,1,1
6,DRAFT,PP,eur,Freedom Academy,1,350000.0,0,0


## 10. Appointment Summary

In [10]:
appointments_summary_df = q("""
SELECT
  a.snapshot_call_category,
  a.source,
  COALESCE(ss.name, 'No Outcome') AS outcome_name,
  COUNT(*) AS appointment_count,
  COUNT(*) FILTER (WHERE a.no_show = true) AS no_show_count,
  COUNT(*) FILTER (WHERE a.no_show = false) AS attended_or_not_marked_no_show_count
FROM appointments a
LEFT JOIN sales_statuses ss
  ON ss.id = a.outcome_id
 AND ss.clerk_org_id = a.clerk_org_id
WHERE a.clerk_org_id = :org_id
  AND a.is_deleted = false
GROUP BY a.snapshot_call_category, a.source, COALESCE(ss.name, 'No Outcome')
ORDER BY appointment_count DESC
""", org_params(), limit=100)

appointments_summary_df

,snapshot_call_category,source,outcome_name,appointment_count,no_show_count,attended_or_not_marked_no_show_count
0,SALES_CALL,CALENDLY,Won,128,2,126
1,SALES_CALL,CALENDLY,Canceled,78,1,77
2,SALES_CALL,CALENDLY,No Sale - Follow Up,76,0,76
3,SALES_CALL,CALENDLY,No Show,49,49,0
4,SALES_CALL,CALENDLY,No Sale - Lost,41,1,40
5,SALES_CALL,CALENDLY,Call Booked,40,0,40
6,SALES_CALL,CALENDLY,Rescheduled,26,1,25
7,SALES_CALL,CALENDLY,No Sale - Unqualified,11,0,11
8,SALES_CALL,CALENDLY,Deposit,2,0,2


## 11. Acquisition Summary

In [11]:
acquisition_summary_df = q("""
SELECT
  oi.source,
  oi.provider_form_name,
  ta.utm_source,
  ta.utm_medium,
  ta.utm_campaign,
  COUNT(*) AS opt_in_count,
  COUNT(DISTINCT oi.lead_id) AS lead_count
FROM opt_ins oi
LEFT JOIN traffic_attributions ta
  ON ta.opt_in_id = oi.id
WHERE oi.clerk_org_id = :org_id
GROUP BY oi.source, oi.provider_form_name, ta.utm_source, ta.utm_medium, ta.utm_campaign
ORDER BY opt_in_count DESC
""", org_params(), limit=100)

acquisition_summary_df

,source,provider_form_name,utm_source,utm_medium,utm_campaign,opt_in_count,lead_count
0,CALENDLY,Qualification Call Freedom Academy (TF),NaN,NaN,NaN,253,246
1,OTHER,Freedom Academy Typeform TBF,youtube,tbf,freedom academy,105,102
2,OTHER,Freedom Academy Typeform Instagram,NaN,NaN,NaN,74,71
3,CALENDLY,Qualification Call Freedom Academy (DMB),NaN,NaN,NaN,67,67
4,OTHER,Freedom Academy Typeform,NaN,NaN,NaN,48,47
5,CALENDLY,Strategy Call - Freedom,NaN,NaN,NaN,40,40
6,OTHER,Freedom Academy Typeform Instagram,instagram,dm,diditaihuttu,28,26
7,CALENDLY,Strategy Call - Freedom - FU,NaN,NaN,NaN,26,23
8,CALENDLY,Strategy Call - Freedom - (FU),NaN,NaN,NaN,25,22
9,CALENDLY,Unknown,NaN,NaN,NaN,22,22


## 12. Lead 360 Starter Query

Replace `lead_email` with a real email from your data.

In [12]:
lead_email = "replace@example.com"

lead_360_starter_df = q("""
SELECT
  l.id AS lead_id,
  l.full_name,
  l.email,
  l.phone_e164,
  l.source,
  ss.name AS current_status,
  l.assigned_to,
  l.setter_id,
  l.next_touch_point_at,
  l.next_touch_point_type,
  COUNT(DISTINCT a.id) AS appointment_count,
  COUNT(DISTINCT c.id) AS contract_count,
  COUNT(DISTINCT p.id) AS payment_count,
  SUM(p.amount) FILTER (WHERE p.status = 'PAID') AS paid_amount,
  SUM(p.amount) FILTER (WHERE p.status = 'PENDING') AS pending_amount
FROM leads l
LEFT JOIN sales_statuses ss
  ON ss.id = l.status_id
 AND ss.clerk_org_id = l.clerk_org_id
LEFT JOIN appointments a
  ON a.lead_id = l.id
 AND a.clerk_org_id = l.clerk_org_id
 AND a.is_deleted = false
LEFT JOIN contracts c
  ON c.lead_id = l.id
 AND c.clerk_org_id = l.clerk_org_id
 AND c.is_deleted = false
LEFT JOIN payments p
  ON p.lead_id = l.id
 AND p.clerk_org_id = l.clerk_org_id
 AND p.is_deleted = false
WHERE l.clerk_org_id = :org_id
  AND l.is_deleted = false
  AND l.email = :lead_email
GROUP BY
  l.id,
  l.full_name,
  l.email,
  l.phone_e164,
  l.source,
  ss.name,
  l.assigned_to,
  l.setter_id,
  l.next_touch_point_at,
  l.next_touch_point_type
""", org_params(lead_email=lead_email), limit=20)

lead_360_starter_df

,lead_id,full_name,email,phone_e164,source,current_status,assigned_to,setter_id,next_touch_point_at,next_touch_point_type,appointment_count,contract_count,payment_count,paid_amount,pending_amount


## 13. Write Your Own Query Here

Keep this pattern: use named params and always include `clerk_org_id = :org_id` for business tables.

In [13]:
custom_df = q("""
SELECT
  COUNT(*) AS active_leads
FROM leads
WHERE clerk_org_id = :org_id
  AND is_deleted = false
""", org_params(), limit=20)

custom_df

,active_leads
0,515
